# Steps 4 & 5: Fetch TextVQA to Colab and Upload to Hugging Face Hub

This notebook executes the ingestion and upload pipeline for **TextVQA (v0.5.1)**:
1. **High-Speed Download:** Fetches train/val image archive (`train_val_images.zip`, ~6.6 GB), test image archive (`test_images.zip`, ~926 MB), annotations, Rosetta OCR v0.2 token files, and OpenImages EXIF rotation CSVs.
2. **Canonical Structuring:** Organizes assets into `images/`, `annotations/`, `ocr/`, `metadata/`.
3. **Dataset Card Generation:** Automatically writes a `README.md` with dataset tags and licenses.
4. **Hugging Face Upload:** Uploads via `HfApi.upload_folder(...)`.
5. **Disk Reclaim & Verification:** Purges local files and lists verified remote files on the Hub.

In [ ]:
# 1. Setup tools and authenticate
!sudo apt-get install -y -qq aria2
!pip install -q --upgrade huggingface_hub

import os
import shutil
from pathlib import Path
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, ModuleNotFoundError):
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found! Please configure HF_TOKEN in Colab Secrets.")

login(token=HF_TOKEN, write_permission=True)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
TEXTVQA_REPO = f"{username}/textvqa-v0.5.1"
print(f"✓ Authenticated as: {username}")
print(f"✓ Target Repo: {TEXTVQA_REPO}")

In [ ]:
# 2. Check initial disk allocation
!df -h /

In [ ]:
# 3. Download TextVQA v0.5.1 files via aria2c
TEXTVQA_RAW = Path("./textvqa_raw")
TEXTVQA_CLEAN = Path("./textvqa_dataset")

shutil.rmtree(TEXTVQA_RAW, ignore_errors=True)
shutil.rmtree(TEXTVQA_CLEAN, ignore_errors=True)
TEXTVQA_RAW.mkdir(parents=True, exist_ok=True)

# Direct URLs for TextVQA v0.5.1 assets
textvqa_urls = """
https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
https://dl.fbaipublicfiles.com/textvqa/images/test_images.zip
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_test.json
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_Rosetta_OCR_v0.2_train.json
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_Rosetta_OCR_v0.2_val.json
https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_Rosetta_OCR_v0.2_test.json
https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv
https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv
"""

with open("textvqa_urls.txt", "w") as f:
    f.write(textvqa_urls.strip())

print(">>> Starting high-speed parallel download with aria2c...")
!aria2c -j 4 -x 8 -s 8 -k 1M --check-certificate=false -i textvqa_urls.txt --dir={TEXTVQA_RAW}

print("Downloaded raw assets:")
!ls -lh {TEXTVQA_RAW}

In [ ]:
# 4. Restructure into canonical layout & generate README.md
(TEXTVQA_CLEAN / "images").mkdir(parents=True, exist_ok=True)
(TEXTVQA_CLEAN / "annotations").mkdir(parents=True, exist_ok=True)
(TEXTVQA_CLEAN / "ocr").mkdir(parents=True, exist_ok=True)
(TEXTVQA_CLEAN / "metadata").mkdir(parents=True, exist_ok=True)

for file in TEXTVQA_RAW.iterdir():
    if file.name.endswith(".zip"):
        shutil.move(str(file), TEXTVQA_CLEAN / "images" / file.name)
    elif "Rosetta_OCR" in file.name:
        shutil.move(str(file), TEXTVQA_CLEAN / "ocr" / file.name)
    elif file.name.endswith(".json"):
        shutil.move(str(file), TEXTVQA_CLEAN / "annotations" / file.name)
    elif file.name.endswith(".csv"):
        shutil.move(str(file), TEXTVQA_CLEAN / "metadata" / file.name)

# Dataset Card
textvqa_readme = f"""---
license: cc-by-4.0
task_categories:
- visual-question-answering
size_categories:
- 10K<n<100K
tags:
- textvqa
- scene-text
- vqa
- multimodal
---

# TextVQA (Version 0.5.1) Dataset Archive

- **Source:** [TextVQA Official Website](https://textvqa.org/dataset/)
- **Paper:** ["Towards VQA Models That Can Read" (Singh et al., CVPR 2019)](https://openaccess.thecvf.com/content_CVPR_2019/html/Singh_Towards_VQA_Models_That_Can_Read_CVPR_2019_paper.html)
- **License:** Creative Commons Attribution 4.0 International (CC BY 4.0)

## Repository Contents
- `images/`: `train_val_images.zip` (25,119 images) and `test_images.zip` (3,289 images)
- `annotations/`: Full question and answers for train, val, and test splits (v0.5.1)
- `ocr/`: Meta AI Rosetta OCR v0.2 token files
- `metadata/`: OpenImages EXIF rotation mapping CSVs
"""

with open(TEXTVQA_CLEAN / "README.md", "w") as f:
    f.write(textvqa_readme.strip())

print("Structured directory:")
!ls -lh {TEXTVQA_CLEAN}/*

In [ ]:
# 5. Upload folder to Hugging Face Hub
print(f">>> Uploading TextVQA to https://huggingface.co/datasets/{TEXTVQA_REPO}...")
api.upload_folder(
    folder_path=str(TEXTVQA_CLEAN),
    repo_id=TEXTVQA_REPO,
    repo_type="dataset",
    commit_message="Ingest TextVQA 0.5.1 images, annotations, Rosetta OCR v0.2, and rotation metadata"
)
print("✓ Upload completed successfully!")

In [ ]:
# 6. Verify remote files on Hugging Face Hub
print(f"=== Files in https://huggingface.co/datasets/{TEXTVQA_REPO} ===")
remote_files = api.list_repo_files(repo_id=TEXTVQA_REPO, repo_type="dataset")
for f in remote_files:
    print(f" - {f}")

In [ ]:
# 7. Purge local disk to reclaim space
print(">>> Purging local files...")
shutil.rmtree(TEXTVQA_RAW, ignore_errors=True)
shutil.rmtree(TEXTVQA_CLEAN, ignore_errors=True)
if os.path.exists("textvqa_urls.txt"):
    os.remove("textvqa_urls.txt")

print("Remaining disk space after purge:")
!df -h /